# N14 — Safety Car Probability Model

This notebook trains a Safety Car probability classifier on the labeled dataset
produced by N13. The model outputs `P(SC within 5 laps)` given the observable
race state at any lap — the core signal consumed by the Strategy Agent to
anticipate full-course caution periods.

**Architecture:** `RandomForestClassifier → Platt calibration → P(SC | lap state)`

**Validation:** temporal split — train 2023+2024 / test 2025.

**Input:** `data/processed/sc_labeled/sc_labeled_2023_2025.parquet`

**Exports:**
- `data/models/safety_car_probability/rf_sc_v1.pkl`
- `data/models/safety_car_probability/calibrator.pkl`
- `data/models/safety_car_probability/model_config.json`


---

## Step 0 — Setup and Imports


In [ ]:
# ── Step 0 — Setup ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
import pathlib
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.special import expit
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap

In [ ]:
# ── Repo root ─────────────────────────────────────────────────────────────────
repo_root = pathlib.Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUTS    = repo_root / "notebooks" / "strategy" / "sc_probability" / "outputs"
PROCESSED  = repo_root / "data" / "processed" / "sc_labeled"
EXPORT_DIR = repo_root / "data" / "models" / "safety_car_probability"

OUTPUTS.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root  : {repo_root}")
print(f"Outputs    : {OUTPUTS}")
print(f"Processed  : {PROCESSED}")
print(f"Export dir : {EXPORT_DIR}")